In [1]:
# ============================================
# STEP 1: Install Required Libraries
# AI HR Copilot - Environment Setup
# ============================================

# We use -q (quiet) to keep output clean, and %%capture is avoided
# so you can still SEE that installs succeeded (beginner-friendly).

!pip install -q PyMuPDF
!pip install -q sentence-transformers
!pip install -q transformers
!pip install -q torch
!pip install -q pandas numpy

print("✅ All libraries installed successfully!")
print("Environment is ready for AI HR Copilot.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 37.1 MB/s eta 0:00:00
✅ All libraries installed successfully!
Environment is ready for AI HR Copilot.


In [2]:
# ============================================
# STEP 2: Upload Resumes
# AI HR Copilot - Resume Upload & Validation
# ============================================

from google.colab import files

def upload_resumes():
    """
    Opens a file picker for the user to upload one or more resume PDFs.
    Returns a dictionary of {filename: file_bytes} for valid PDFs only.
    """
    print("📤 Please select one or more resume PDF files to upload...")
    uploaded = files.upload()  # Opens browser file picker

    valid_resumes = {}
    invalid_files = []

    for filename, file_bytes in uploaded.items():
        if is_valid_pdf(filename, file_bytes):
            valid_resumes[filename] = file_bytes
        else:
            invalid_files.append(filename)

    # Report results clearly
    print(f"\n✅ Successfully uploaded {len(valid_resumes)} valid resume(s):")
    for name in valid_resumes:
        print(f"   - {name}")

    if invalid_files:
        print(f"\n⚠️ Skipped {len(invalid_files)} invalid file(s):")
        for name in invalid_files:
            print(f"   - {name} (not a valid PDF or empty)")

    if not valid_resumes:
        print("\n❌ No valid resumes uploaded. Please try again.")

    return valid_resumes


def is_valid_pdf(filename, file_bytes):
    """
    Validates that a file is a non-empty PDF.
    Checks both file extension AND PDF file signature (magic bytes).
    """
    # Check extension
    if not filename.lower().endswith(".pdf"):
        return False

    # Check file isn't empty
    if len(file_bytes) == 0:
        return False

    # Check PDF signature (real PDFs start with "%PDF-")
    if not file_bytes[:5] == b"%PDF-":
        return False

    return True


# Run the upload process
resume_files = upload_resumes()

📤 Please select one or more resume PDF files to upload...


Saving resume_priya_menon.pdf to resume_priya_menon.pdf
Saving resume_rohit_sharma.pdf to resume_rohit_sharma.pdf
Saving resume_ananya_rao.pdf to resume_ananya_rao.pdf

✅ Successfully uploaded 3 valid resume(s):
   - resume_priya_menon.pdf
   - resume_rohit_sharma.pdf
   - resume_ananya_rao.pdf


In [3]:
# ============================================
# STEP 3: Upload Job Description
# AI HR Copilot - Job Description Upload & Validation
# ============================================

from google.colab import files

def upload_job_description():
    """
    Opens a file picker for the user to upload exactly ONE job description PDF.
    Returns a tuple: (filename, file_bytes) or (None, None) if invalid.
    """
    print("📤 Please select ONE Job Description PDF to upload...")
    uploaded = files.upload()  # Opens browser file picker

    if len(uploaded) == 0:
        print("❌ No file uploaded. Please try again.")
        return None, None

    if len(uploaded) > 1:
        print(f"⚠️ You uploaded {len(uploaded)} files, but only ONE Job Description is allowed.")
        print("   Using the first valid PDF found and ignoring the rest.")

    # Reuse the same validator function from Step 2 (no duplicate logic)
    for filename, file_bytes in uploaded.items():
        if is_valid_pdf(filename, file_bytes):
            print(f"\n✅ Job Description uploaded successfully: {filename}")
            return filename, file_bytes
        else:
            print(f"\n❌ '{filename}' is not a valid PDF. Please try again.")

    return None, None


# Run the upload process
# NOTE: is_valid_pdf() was already defined in Step 2 — we reuse it here
jd_filename, jd_bytes = upload_job_description()

📤 Please select ONE Job Description PDF to upload...


Saving job_description.pdf to job_description.pdf

✅ Job Description uploaded successfully: job_description.pdf


In [4]:
# ============================================
# STEP 4: Extract PDF Text
# AI HR Copilot - PDF Text Extraction
# ============================================

import fitz  # PyMuPDF

def extract_text_from_pdf(filename, file_bytes):
    """
    Extracts all text from a PDF file given as raw bytes.
    Returns extracted text as a string, or None if extraction fails.
    """
    try:
        # Open PDF directly from memory (no need to save to disk)
        pdf_document = fitz.open(stream=file_bytes, filetype="pdf")

        extracted_text = ""
        for page_number in range(len(pdf_document)):
            page = pdf_document[page_number]
            extracted_text += page.get_text()

        pdf_document.close()

        # Warn if extraction returned almost nothing (likely a scanned/image PDF)
        if len(extracted_text.strip()) < 20:
            print(f"⚠️ '{filename}' produced very little text — it may be a scanned/image-based PDF.")

        return extracted_text

    except Exception as e:
        print(f"❌ Failed to extract text from '{filename}': {e}")
        return None


def extract_all_resumes(resume_files):
    """
    Applies text extraction to every resume in the resume_files dict.
    Returns a dict: {filename: extracted_text}
    """
    resume_texts = {}
    for filename, file_bytes in resume_files.items():
        text = extract_text_from_pdf(filename, file_bytes)
        if text:  # Only keep successful extractions
            resume_texts[filename] = text
            print(f"✅ Extracted text from: {filename} ({len(text)} characters)")
        else:
            print(f"⚠️ Skipping '{filename}' — no text extracted.")

    return resume_texts


# Extract text from all resumes (from Step 2)
resume_texts = extract_all_resumes(resume_files)

# Extract text from the job description (from Step 3)
jd_text = extract_text_from_pdf(jd_filename, jd_bytes)

print(f"\n📄 Job Description text length: {len(jd_text) if jd_text else 0} characters")

✅ Extracted text from: resume_priya_menon.pdf (625 characters)
✅ Extracted text from: resume_rohit_sharma.pdf (731 characters)
✅ Extracted text from: resume_ananya_rao.pdf (853 characters)

📄 Job Description text length: 860 characters


In [5]:
# ============================================
# STEP 5: Clean Text
# AI HR Copilot - Text Cleaning & Normalization
# ============================================

import re

def clean_text(raw_text):
    """
    Cleans raw extracted PDF text for NLP processing.
    - Removes control/non-printable characters
    - Normalizes whitespace and line breaks
    - Strips common bullet/symbol clutter
    Returns a cleaned string. Does NOT lowercase (case can matter for names/skills).
    """
    if not raw_text:
        return ""

    text = raw_text

    # Remove non-printable/control characters (keep standard punctuation & unicode letters)
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", " ", text)

    # Replace common bullet symbols with a simple dash for consistency
    text = re.sub(r"[•●▪◦‣∙]", "-", text)

    # Collapse multiple spaces/tabs into a single space
    text = re.sub(r"[ \t]+", " ", text)

    # Collapse 3+ blank lines into a single blank line
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Trim leading/trailing whitespace on each line
    text = "\n".join(line.strip() for line in text.split("\n"))

    # Final trim of the whole string
    text = text.strip()

    return text


def clean_all_resumes(resume_texts):
    """
    Applies clean_text() to every resume's extracted text.
    Returns a dict: {filename: cleaned_text}
    """
    cleaned = {}
    for filename, raw_text in resume_texts.items():
        cleaned_text = clean_text(raw_text)
        cleaned[filename] = cleaned_text
        print(f"✅ Cleaned: {filename} ({len(raw_text)} → {len(cleaned_text)} characters)")

    return cleaned


# Clean all resumes (from Step 4), keep originals untouched
resume_texts_clean = clean_all_resumes(resume_texts)

# Clean the job description (from Step 4)
jd_text_clean = clean_text(jd_text)

print(f"\n📄 Job Description cleaned: {len(jd_text)} → {len(jd_text_clean)} characters")

✅ Cleaned: resume_priya_menon.pdf (625 → 624 characters)
✅ Cleaned: resume_rohit_sharma.pdf (731 → 730 characters)
✅ Cleaned: resume_ananya_rao.pdf (853 → 852 characters)

📄 Job Description cleaned: 860 → 859 characters


In [6]:
# ============================================
# STEP 6: Resume Information Extraction
# AI HR Copilot - Structured Candidate Data Extraction
# ============================================

import re

# Known section header variants we look for in resumes.
# Maps a "canonical" field name -> list of header text variants to detect.
SECTION_HEADERS = {
    "skills": ["skills", "technical skills", "key skills"],
    "education": ["education"],
    "experience": ["experience", "work experience", "professional experience"],
    "projects": ["projects", "academic projects"],
    "certifications": ["certifications", "certificates", "certification"],
}


def extract_email(text):
    """Finds the first email address in the text using regex."""
    match = re.search(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}", text)
    return match.group(0) if match else "Not found"


def extract_phone(text):
    """Finds the first phone number pattern (supports +91, dashes, spaces)."""
    match = re.search(r"(\+?\d{1,3}[-\s]?)?\d{5}[-\s]?\d{5}", text)
    return match.group(0).strip() if match else "Not found"


def extract_name(text):
    """
    Assumes the candidate's name is the first non-empty line of the resume.
    This is a common, reliable heuristic for standard resume layouts.
    """
    for line in text.split("\n"):
        line = line.strip()
        if line:
            return line
    return "Not found"


def extract_section(text, header_variants):
    """
    Finds a section by header (case-insensitive) and returns the text
    until the next known section header (or end of document).
    """
    lines = text.split("\n")
    all_known_headers = [h for variants in SECTION_HEADERS.values() for h in variants]

    start_index = None
    for i, line in enumerate(lines):
        line_lower = line.strip().lower().rstrip(":")
        if line_lower in header_variants:
            start_index = i + 1
            break

    if start_index is None:
        return "Not found"

    # Collect lines until we hit another known section header
    section_lines = []
    for line in lines[start_index:]:
        line_lower = line.strip().lower().rstrip(":")
        if line_lower in all_known_headers:
            break
        section_lines.append(line)

    section_text = "\n".join(section_lines).strip()
    return section_text if section_text else "Not found"


def extract_candidate_info(filename, text):
    """
    Runs all extractors on a single resume's text and returns a structured dict.
    """
    return {
        "filename": filename,
        "name": extract_name(text),
        "email": extract_email(text),
        "phone": extract_phone(text),
        "skills": extract_section(text, SECTION_HEADERS["skills"]),
        "education": extract_section(text, SECTION_HEADERS["education"]),
        "experience": extract_section(text, SECTION_HEADERS["experience"]),
        "projects": extract_section(text, SECTION_HEADERS["projects"]),
        "certifications": extract_section(text, SECTION_HEADERS["certifications"]),
    }


def extract_all_candidates(resume_texts_clean):
    """
    Applies extract_candidate_info() to every resume.
    Returns a list of candidate info dictionaries.
    """
    candidates = []
    for filename, text in resume_texts_clean.items():
        info = extract_candidate_info(filename, text)
        candidates.append(info)
        print(f"✅ Extracted structured info for: {info['name']} ({filename})")

    return candidates


# Extract structured info from all cleaned resumes (from Step 5)
candidates_data = extract_all_candidates(resume_texts_clean)

# Preview the first candidate's extracted info
print("\n--- Sample Extracted Candidate ---")
for key, value in candidates_data[0].items():
    print(f"{key.upper()}: {value}\n")

✅ Extracted structured info for: Priya Menon (resume_priya_menon.pdf)
✅ Extracted structured info for: Rohit Sharma (resume_rohit_sharma.pdf)
✅ Extracted structured info for: Ananya Rao (resume_ananya_rao.pdf)

--- Sample Extracted Candidate ---
FILENAME: resume_priya_menon.pdf

NAME: Priya Menon

EMAIL: priya.menon@example.com

PHONE: +91-99887-76655

SKILLS: HTML, CSS, JavaScript, PHP, MySQL, WordPress

EDUCATION: B.Sc in Computer Applications, St. Xavier College, 2023 - CGPA 7.2

EXPERIENCE: Web Development Intern, PixelWorks Studio (Mar 2022 - Aug 2022)
- Built responsive websites using HTML, CSS and JavaScript
- Managed WordPress based client websites
- Fixed cross-browser compatibility issues

PROJECTS: - E-commerce website using PHP and MySQL
- Portfolio website builder using WordPress themes
- Simple to-do list app using JavaScript

CERTIFICATIONS: Responsive Web Design (freeCodeCamp)



In [7]:
# ============================================
# STEP 7: Generate Embeddings
# AI HR Copilot - Semantic Embeddings with Sentence Transformers
# ============================================

from sentence_transformers import SentenceTransformer

# Load the embedding model once (this line is slow ~10-20 sec, encoding after is fast)
print("⏳ Loading embedding model (all-MiniLM-L6-v2)... this may take a moment.")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ Embedding model loaded.")


def build_candidate_profile_text(candidate):
    """
    Combines the most job-relevant fields (Skills, Experience, Projects)
    into a single text block for embedding.
    Education/Certifications are excluded here — they're handled
    separately in Section 10 (Skill Gap Analysis) and reporting.
    """
    parts = [
        candidate.get("skills", ""),
        candidate.get("experience", ""),
        candidate.get("projects", ""),
    ]
    # Filter out "Not found" placeholders so they don't pollute the embedding
    parts = [p for p in parts if p and p != "Not found"]
    return " ".join(parts)


def generate_embeddings(candidates_data, jd_text_clean, model):
    """
    Generates embeddings for the job description and every candidate's profile text.
    Adds an 'embedding' key and a 'profile_text' key to each candidate dict.
    Returns the JD embedding separately.
    """
    # Encode the Job Description once
    jd_embedding = model.encode(jd_text_clean, convert_to_tensor=True)

    for candidate in candidates_data:
        profile_text = build_candidate_profile_text(candidate)

        if not profile_text.strip():
            print(f"⚠️ '{candidate['filename']}' has no usable Skills/Experience/Projects text — "
                  f"embedding may be unreliable.")
            profile_text = candidate.get("name", "")  # fallback so encode() doesn't fail on empty string

        candidate["profile_text"] = profile_text
        candidate["embedding"] = model.encode(profile_text, convert_to_tensor=True)
        print(f"✅ Generated embedding for: {candidate['name']}")

    return jd_embedding


# Generate embeddings for all candidates (from Step 6) and the JD (from Step 5)
jd_embedding = generate_embeddings(candidates_data, jd_text_clean, embedding_model)

print(f"\n📄 JD embedding shape: {jd_embedding.shape}")
print(f"👤 Sample candidate embedding shape: {candidates_data[0]['embedding'].shape}")

⏳ Loading embedding model (all-MiniLM-L6-v2)... this may take a moment.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded.
✅ Generated embedding for: Priya Menon
✅ Generated embedding for: Rohit Sharma
✅ Generated embedding for: Ananya Rao

📄 JD embedding shape: torch.Size([384])
👤 Sample candidate embedding shape: torch.Size([384])


In [8]:
# ============================================
# STEP 8: Similarity Matching
# AI HR Copilot - Cosine Similarity & Match Score Calculation
# ============================================

from sentence_transformers import util


def calculate_match_score(candidate_embedding, jd_embedding):
    """
    Computes cosine similarity between a candidate's embedding and the JD embedding.
    Returns a match score as a percentage (0-100), rounded to 1 decimal place.
    """
    similarity = util.cos_sim(candidate_embedding, jd_embedding)
    # cos_sim returns a 2D tensor (1x1); extract the single float value
    similarity_score = similarity.item()

    # Clip negative similarities to 0 (can rarely happen with unrelated text)
    similarity_score = max(similarity_score, 0)

    # Convert to a 0-100 percentage scale
    match_percentage = round(similarity_score * 100, 1)
    return match_percentage


def calculate_all_match_scores(candidates_data, jd_embedding):
    """
    Applies calculate_match_score() to every candidate.
    Adds a 'match_score' key to each candidate's dict.
    """
    for candidate in candidates_data:
        score = calculate_match_score(candidate["embedding"], jd_embedding)
        candidate["match_score"] = score
        print(f"✅ {candidate['name']}: {score}% match")

    return candidates_data


# Calculate match scores for all candidates (embeddings from Step 7)
candidates_data = calculate_all_match_scores(candidates_data, jd_embedding)

✅ Priya Menon: 21.4% match
✅ Rohit Sharma: 48.6% match
✅ Ananya Rao: 59.2% match


In [9]:
# ============================================
# STEP 9: Candidate Ranking
# AI HR Copilot - Rank Candidates by Match Score
# ============================================

def rank_candidates(candidates_data):
    """
    Sorts candidates by match_score (descending) and assigns a 'rank' field.
    Returns the sorted list (does not mutate original order outside this list).
    """
    # Sort by match_score, highest first
    ranked = sorted(candidates_data, key=lambda c: c["match_score"], reverse=True)

    # Assign rank numbers (1 = best match)
    for position, candidate in enumerate(ranked, start=1):
        candidate["rank"] = position

    return ranked


def display_ranking(ranked_candidates):
    """
    Prints a clean, readable ranking table to the console.
    """
    print("=" * 50)
    print("CANDIDATE RANKING")
    print("=" * 50)
    for candidate in ranked_candidates:
        print(f"#{candidate['rank']}  {candidate['name']:<20} "
              f"Match: {candidate['match_score']}%")
    print("=" * 50)


# Rank all candidates (match scores from Step 8)
candidates_data = rank_candidates(candidates_data)

# Display the ranking
display_ranking(candidates_data)

CANDIDATE RANKING
#1  Ananya Rao           Match: 59.2%
#2  Rohit Sharma         Match: 48.6%
#3  Priya Menon          Match: 21.4%


In [10]:
# ============================================
# STEP 10: Skill Gap Analysis
# AI HR Copilot - Matching & Missing Skills Detection
# ============================================

import re


def extract_jd_skills(jd_text_clean):
    """
    Extracts a clean list of individual skills from the Job Description text.
    Looks for 'Required Skills' and 'Preferred Skills' sections and splits
    their contents on commas.
    """
    skill_sections = ["required skills", "preferred skills"]
    lines = jd_text_clean.split("\n")

    all_skills = []
    capture = False

    for line in lines:
        line_clean = line.strip()
        line_lower = line_clean.lower().rstrip(":")

        if line_lower in skill_sections:
            capture = True
            continue

        # Stop capturing once we hit a new section header (a short line ending in nothing,
        # commonly styled differently) — heuristic: a line with no commas AND is short AND
        # doesn't look like a skill list signals a new section.
        if capture:
            if line_clean == "" or (len(line_clean.split(",")) == 1 and len(line_clean) < 40
                                     and not any(c.isupper() for c in line_clean[1:])):
                # Looks like a new header or blank line -> stop capturing after this line
                if "," in line_clean:
                    all_skills.extend([s.strip() for s in line_clean.split(",")])
                capture = False
                continue
            all_skills.extend([s.strip() for s in line_clean.split(",")])

    # Clean up: remove empties, deduplicate while preserving order
    seen = set()
    cleaned_skills = []
    for skill in all_skills:
        skill = skill.strip(" .")
        if skill and skill.lower() not in seen:
            seen.add(skill.lower())
            cleaned_skills.append(skill)

    return cleaned_skills


def analyze_skill_gap(candidate_full_text, jd_skills):
    """
    Checks which JD skills appear in the candidate's resume text (case-insensitive,
    whole-word matching to avoid partial-word false positives like 'R' matching inside 'Learning').
    Returns (matching_skills, missing_skills).
    """
    matching_skills = []
    missing_skills = []

    text_lower = candidate_full_text.lower()

    for skill in jd_skills:
        skill_lower = skill.lower()
        # Use word-boundary regex so "SQL" doesn't match inside "MySQL2" unexpectedly,
        # and short skills like "R" or "Git" aren't falsely triggered by substrings.
        pattern = r"\b" + re.escape(skill_lower) + r"\b"
        if re.search(pattern, text_lower):
            matching_skills.append(skill)
        else:
            missing_skills.append(skill)

    return matching_skills, missing_skills


def run_skill_gap_analysis(candidates_data, resume_texts_clean, jd_text_clean):
    """
    Runs skill gap analysis for every candidate and stores results in their dict.
    """
    jd_skills = extract_jd_skills(jd_text_clean)
    print(f"📋 JD Skills detected ({len(jd_skills)}): {', '.join(jd_skills)}\n")

    for candidate in candidates_data:
        full_text = resume_texts_clean.get(candidate["filename"], "")
        matching, missing = analyze_skill_gap(full_text, jd_skills)

        candidate["matching_skills"] = matching
        candidate["missing_skills"] = missing

        print(f"👤 {candidate['name']}")
        print(f"   ✓ Matching: {', '.join(matching) if matching else 'None'}")
        print(f"   ✗ Missing:  {', '.join(missing) if missing else 'None'}\n")

    return jd_skills


# Run skill gap analysis (uses cleaned texts from Step 5, candidates from Step 9)
jd_skills = run_skill_gap_analysis(candidates_data, resume_texts_clean, jd_text_clean)

📋 JD Skills detected (13): Python, Machine Learning, PyTorch, Deep Learning, SQL, Data Structures, Git, Docker, Linux, AWS, NLP, Computer Vision, REST APIs

👤 Ananya Rao
   ✓ Matching: Python, Machine Learning, PyTorch, Deep Learning, SQL, Data Structures, Git, Docker, Linux, AWS, NLP
   ✗ Missing:  Computer Vision, REST APIs

👤 Rohit Sharma
   ✓ Matching: Python, Machine Learning, SQL, Data Structures, Git, REST APIs
   ✗ Missing:  PyTorch, Deep Learning, Docker, Linux, AWS, NLP, Computer Vision

👤 Priya Menon
   ✓ Matching: None
   ✗ Missing:  Python, Machine Learning, PyTorch, Deep Learning, SQL, Data Structures, Git, Docker, Linux, AWS, NLP, Computer Vision, REST APIs



In [11]:
# ============================================
# STEP 11: Explainable AI
# AI HR Copilot - Human-Readable Score Explanations
# ============================================

def generate_skill_checklist(matching_skills, missing_skills):
    """
    Builds a ✓/✗ checklist string from matching and missing skills,
    matching the format specified in the project spec.
    """
    lines = []
    for skill in matching_skills:
        lines.append(f"✓ {skill} matches")
    for skill in missing_skills:
        lines.append(f"✗ {skill} missing")
    return "\n".join(lines) if lines else "No skill data available"


def identify_strengths(candidate):
    """
    Rule-based strength identification using matching skills count
    and presence of relevant Experience/Projects text.
    """
    strengths = []
    match_count = len(candidate.get("matching_skills", []))

    if match_count >= 6:
        strengths.append("Strong technical skill overlap with job requirements")
    elif match_count >= 3:
        strengths.append("Reasonable technical skill overlap with job requirements")

    if candidate.get("experience", "Not found") != "Not found":
        strengths.append("Has relevant hands-on work experience")

    if candidate.get("projects", "Not found") != "Not found":
        strengths.append("Has relevant project experience demonstrating applied skills")

    return strengths if strengths else ["No standout strengths identified from available data"]


def identify_weaknesses(candidate):
    """
    Rule-based weakness identification based on missing skill count
    and match score.
    """
    weaknesses = []
    missing_count = len(candidate.get("missing_skills", []))
    score = candidate.get("match_score", 0)

    if missing_count >= 5:
        weaknesses.append(f"Missing {missing_count} key required/preferred skills")
    elif missing_count > 0:
        weaknesses.append(f"Missing a few skills: {', '.join(candidate['missing_skills'][:3])}")

    if score < 40:
        weaknesses.append("Overall profile shows low semantic alignment with the job description")

    return weaknesses if weaknesses else ["No major weaknesses identified"]


def generate_explanation(candidate):
    """
    Combines checklist + strengths + weaknesses into one explanation dict,
    and stores it back into the candidate record.
    """
    checklist = generate_skill_checklist(
        candidate.get("matching_skills", []),
        candidate.get("missing_skills", [])
    )
    strengths = identify_strengths(candidate)
    weaknesses = identify_weaknesses(candidate)

    candidate["checklist"] = checklist
    candidate["strengths"] = strengths
    candidate["weaknesses"] = weaknesses

    return candidate


def run_explainability(candidates_data):
    """
    Applies generate_explanation() to every candidate and prints a preview.
    """
    for candidate in candidates_data:
        generate_explanation(candidate)

        print("=" * 50)
        print(f"{candidate['name']}  —  Match Score: {candidate['match_score']}%")
        print("-" * 50)
        print("Reason:")
        print(candidate["checklist"])
        print("\nStrengths:")
        for s in candidate["strengths"]:
            print(f"  + {s}")
        print("\nWeaknesses:")
        for w in candidate["weaknesses"]:
            print(f"  - {w}")
        print("=" * 50 + "\n")

    return candidates_data


# Generate explanations for all candidates (skill gap data from Step 10)
candidates_data = run_explainability(candidates_data)

Ananya Rao  —  Match Score: 59.2%
--------------------------------------------------
Reason:
✓ Python matches
✓ Machine Learning matches
✓ PyTorch matches
✓ Deep Learning matches
✓ SQL matches
✓ Data Structures matches
✓ Git matches
✓ Docker matches
✓ Linux matches
✓ AWS matches
✓ NLP matches
✗ Computer Vision missing
✗ REST APIs missing

Strengths:
  + Strong technical skill overlap with job requirements
  + Has relevant hands-on work experience
  + Has relevant project experience demonstrating applied skills

Weaknesses:
  - Missing a few skills: Computer Vision, REST APIs

Rohit Sharma  —  Match Score: 48.6%
--------------------------------------------------
Reason:
✓ Python matches
✓ Machine Learning matches
✓ SQL matches
✓ Data Structures matches
✓ Git matches
✓ REST APIs matches
✗ PyTorch missing
✗ Deep Learning missing
✗ Docker missing
✗ Linux missing
✗ AWS missing
✗ NLP missing
✗ Computer Vision missing

Strengths:
  + Strong technical skill overlap with job requirements
  + Ha

In [13]:
# ============================================
# STEP 12: HR Decision Engine
# AI HR Copilot - Rule-Based Hiring Recommendation
# ============================================

# Thresholds are defined as constants so they're easy to find and tune later.
STRONGLY_RECOMMEND_SCORE = 65
RECOMMEND_SCORE = 45
CONSIDER_SCORE = 25

STRONGLY_RECOMMEND_SKILL_RATIO = 0.6
RECOMMEND_SKILL_RATIO = 0.4


def calculate_skill_ratio(candidate):
    """
    Calculates the fraction of JD skills the candidate matches (0.0 - 1.0).
    Returns 0.0 safely if no skill data exists (avoids division by zero).
    """
    matching = len(candidate.get("matching_skills", []))
    missing = len(candidate.get("missing_skills", []))
    total = matching + missing

    if total == 0:
        return 0.0

    return matching / total


def make_recommendation(candidate):
    """
    Combines match_score and skill_ratio into a final hiring recommendation.
    Rule-based and fully explainable — no black-box model involved.
    """
    score = candidate.get("match_score", 0)
    skill_ratio = calculate_skill_ratio(candidate)

    if score >= STRONGLY_RECOMMEND_SCORE and skill_ratio >= STRONGLY_RECOMMEND_SKILL_RATIO:
        recommendation = "Strongly Recommend"
    elif score >= RECOMMEND_SCORE and skill_ratio >= RECOMMEND_SKILL_RATIO:
        recommendation = "Recommend"
    elif score >= CONSIDER_SCORE:
        recommendation = "Consider"
    else:
        recommendation = "Reject"

    candidate["skill_ratio"] = round(skill_ratio, 2)
    candidate["recommendation"] = recommendation
    return candidate


def run_hr_decision_engine(candidates_data):
    """
    Applies make_recommendation() to every candidate and prints a summary.
    """
    for candidate in candidates_data:
        make_recommendation(candidate)
        print(f"👤 {candidate['name']:<20} "
              f"Score: {candidate['match_score']}%  "
              f"Skill Match: {int(candidate['skill_ratio']*100)}%  "
              f"→ {candidate['recommendation']}")

    return candidates_data


# Run the HR decision engine (match scores from Step 8, skill gaps from Step 10)
candidates_data = run_hr_decision_engine(candidates_data)

👤 Ananya Rao           Score: 59.2%  Skill Match: 85%  → Recommend
👤 Rohit Sharma         Score: 48.6%  Skill Match: 46%  → Recommend
👤 Priya Menon          Score: 21.4%  Skill Match: 0%  → Reject


In [14]:
# ============================================
# STEP 15: Final Dashboard
# AI HR Copilot - Final HR Report
# ============================================

def display_candidate_report(candidate):
    """
    Displays one candidate's full HR report in the spec's required format.
    """
    print("=" * 60)
    print(f"CANDIDATE: {candidate['name']}")
    print("=" * 60)
    print(f"Resume Match Score : {candidate['match_score']}%")
    print(f"Matching Skills    : {', '.join(candidate['matching_skills']) if candidate['matching_skills'] else 'None'}")
    print(f"Missing Skills     : {', '.join(candidate['missing_skills']) if candidate['missing_skills'] else 'None'}")
    print(f"\nStrengths:")
    for s in candidate["strengths"]:
        print(f"  + {s}")
    print(f"\nWeaknesses:")
    for w in candidate["weaknesses"]:
        print(f"  - {w}")
    print(f"\nInterview Recommendation : {candidate['recommendation']}")
    print("=" * 60 + "\n")


def display_final_ranking_table(candidates_data):
    """
    Displays a compact ranked comparison table across all candidates.
    """
    print("=" * 60)
    print("RANK ALL CANDIDATES")
    print("=" * 60)
    print(f"{'Rank':<6}{'Name':<20}{'Match %':<10}{'Recommendation':<20}")
    print("-" * 60)
    for c in candidates_data:
        print(f"{c['rank']:<6}{c['name']:<20}{c['match_score']:<10}{c['recommendation']:<20}")
    print("=" * 60)


def display_ai_hr_copilot_dashboard(candidates_data):
    """
    Runs the full Final HR Report: detailed per-candidate cards + ranking summary.
    """
    print("\n" + "#" * 60)
    print("AI HR COPILOT — FINAL REPORT")
    print("#" * 60 + "\n")

    for candidate in candidates_data:
        display_candidate_report(candidate)

    display_final_ranking_table(candidates_data)


# Run the full dashboard (candidates_data now has every field from Steps 6-13)
display_ai_hr_copilot_dashboard(candidates_data)


############################################################
AI HR COPILOT — FINAL REPORT
############################################################

CANDIDATE: Ananya Rao
Resume Match Score : 59.2%
Matching Skills    : Python, Machine Learning, PyTorch, Deep Learning, SQL, Data Structures, Git, Docker, Linux, AWS, NLP
Missing Skills     : Computer Vision, REST APIs

Strengths:
  + Strong technical skill overlap with job requirements
  + Has relevant hands-on work experience
  + Has relevant project experience demonstrating applied skills

Weaknesses:
  - Missing a few skills: Computer Vision, REST APIs

Interview Recommendation : Recommend

CANDIDATE: Rohit Sharma
Resume Match Score : 48.6%
Matching Skills    : Python, Machine Learning, SQL, Data Structures, Git, REST APIs
Missing Skills     : PyTorch, Deep Learning, Docker, Linux, AWS, NLP, Computer Vision

Strengths:
  + Strong technical skill overlap with job requirements
  + Has relevant hands-on work experience
  + Has relevan